# Vinyl grader — Colab GPU (Workflow A + tune + pipeline)

**Run all** top to bottom after:
- **Runtime → Change runtime type → GPU** (Colab Pro)
- Prefer **Python 3.12** runtime (matches `requires-python` in `pyproject.toml`).
- **Colab Secrets**: add `MLFLOW_TRACKING_URI` (and optional `HF_TOKEN`, `GITHUB_TOKEN` for private clone).
- **No `.env` in git** — secrets go in Colab Secrets or a private Drive copy of `.env` copied to the repo root (see cell 2).

**Naming:** This notebook is `colab_vinyl_grader_train.ipynb`. Use `colab_<component>_train.ipynb` for recommender / price_estimator later.

**Flow:** clone → install → Drive artifacts → **transformer_tune** sweep → pick best → merge YAML → **promote** → **`pipeline train`** (with `--skip-baseline`).

**Registration:** `transformer_tune` registers the best preset to MLflow by default (`--register`). The final `pipeline train` may register again — use `--no-register` on the pipeline if you only want the tune registration.

## 1 — Secrets and remote MLflow

Set `os.environ` **before** any `grader` imports that configure MLflow. `load_project_dotenv()` does not overwrite existing variables.

In [ ]:
import os

try:
    from google.colab import userdata
    uri = userdata.get("MLFLOW_TRACKING_URI")
    if uri:
        os.environ["MLFLOW_TRACKING_URI"] = uri
except ImportError:
    print("Not on Colab — set MLFLOW_TRACKING_URI in the shell or rely on repo .env")

assert os.environ.get("MLFLOW_TRACKING_URI", "").strip(), (
    "Set MLFLOW_TRACKING_URI (Colab Secret userdata) or export before running"
)
print("MLFLOW_TRACKING_URI is set (length)", len(os.environ["MLFLOW_TRACKING_URI"]))

# Optional: metrics-only pipeline train (no MLflow artifact uploads)
# os.environ["VINYL_GRADER_MLFLOW_METRICS_ONLY"] = "1"

## 2 — Optional: copy `.env` from Drive to repo root

Uncomment if you keep a private `vinyl_management_system.env` on Drive instead of per-key secrets.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copy(
#     "/content/drive/MyDrive/secure/vinyl_management_system.env",
#     "/content/vinyl_management_system/.env",
# )
# print("Copied .env to repo root")
pass

## 3 — GPU check (runtime-check)

Expect `torch.cuda.is_available()` **True** on a GPU runtime.

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "cuda?", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch Runtime → GPU"

## 4 — Clone repo from GitHub (deps-install)

Replace `YOUR_ORG/YOUR_REPO` and branch. For private repos, use `https://<TOKEN>@github.com/...` from a Secret or `git clone` with `GITHUB_TOKEN`.

In [ ]:
%cd /content
REPO = "vinyl_management_system"
REPO_URL = "https://github.com/YOUR_ORG/YOUR_REPO.git"  # TODO: edit
BRANCH = "main"  # TODO: edit

import os
import subprocess

if not os.path.isdir(f"/content/{REPO}"):
    subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
else:
    print("Repo dir exists; skip clone")

subprocess.run(["git", "-C", f"/content/{REPO}", "fetch", "--all"], check=False)
subprocess.run(["git", "-C", f"/content/{REPO}", "checkout", BRANCH], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", f"/content/{REPO}", "rev-parse", "HEAD"], text=True).strip())

## 5 — Install package (keep Colab CUDA PyTorch)

If `pip install -e .` tries to replace `torch` with a CPU wheel, install dependencies **excluding** `torch` / `torchvision` / `torchaudio`, then `pip install -e . --no-deps`.

In [ ]:
%cd /content/vinyl_management_system
import sys
ROOT = "/content/vinyl_management_system"
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Example: editable install (adjust if torch conflicts)
!pip install -q -e ".[test]" || pip install -q transformers datasets mlflow scikit-learn pyyaml python-dotenv
import torch
print("cuda after install:", torch.cuda.is_available())

## 6 — Mount Drive and unzip artifacts (sync-artifacts)

Zip from local should contain **`paths.splits`** and **`paths.artifacts`** per `grader/configs/grader.yaml` (e.g. `grader/data/.../splits`, `grader/artifacts` including `features/`, `baseline_*_calibrated.pkl`, vectorizers, encoders).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import zipfile
from pathlib import Path

ZIP = Path("/content/drive/MyDrive/path/to/grader_colab_artifacts.zip")  # TODO
assert ZIP.is_file(), f"Missing zip: {ZIP}"

ROOT = Path("/content/vinyl_management_system")
with zipfile.ZipFile(ZIP, "r") as z:
    z.extractall(ROOT)
print("Extracted into", ROOT)

## 7 — Hyperparameter sweep (`transformer_tune`)

Uses local tuning MLflow DB by default during presets; **`--register`** logs best preset to **remote** (`MLFLOW_TRACKING_URI`).

Use **`--mlflow-no-artifacts`** on the sweep to reduce remote artifact noise (metrics still logged where enabled).

For full **`pipeline train`** runs, you can also set **`os.environ["VINYL_GRADER_MLFLOW_METRICS_ONLY"] = "1"`** (in cell 2) so the pipeline forces **`mlflow.log_artifacts`** off even without the CLI flag.

In [ ]:
%cd /content/vinyl_management_system
!python -m grader.src.models.transformer_tune \
  --config grader/configs/grader.yaml \
  --tuning-config grader/configs/transformer_tune.yaml \
  --presets all

## 8 — Best preset from CSV + merged config + promote

In [ ]:
import csv
import copy
import json
from pathlib import Path

import yaml

ROOT = Path("/content/vinyl_management_system")
csv_path = ROOT / "grader" / "reports" / "transformer_tune_results.csv"
assert csv_path.is_file(), f"Run tune first: {csv_path}"

rows = list(csv.DictReader(csv_path.open()))
assert rows
best = max(rows, key=lambda r: float(r["test_mean_macro_f1"]))
BEST_PRESET = best["preset"]
print("BEST_PRESET", BEST_PRESET, "test_mean_macro_f1", best["test_mean_macro_f1"])

base_cfg = yaml.safe_load((ROOT / "grader" / "configs" / "grader.yaml").read_text())
tune_cfg = yaml.safe_load((ROOT / "grader" / "configs" / "transformer_tune.yaml").read_text())
presets = tune_cfg.get("presets") or {}
spec = copy.deepcopy(presets[BEST_PRESET])
spec.pop("description", None)
merged = copy.deepcopy(base_cfg)
mt = merged.setdefault("models", {}).setdefault("transformer", {})
for k, v in spec.items():
    mt[k] = v

merged_path = Path("/tmp/colab_grader_merged.yaml")
merged_path.write_text(yaml.safe_dump(merged, sort_keys=False), encoding="utf-8")
print("Wrote", merged_path)

import subprocess
subprocess.run(
    ["python", "-m", "grader.src.models.transformer_tune", "--config", "grader/configs/grader.yaml", "--promote", BEST_PRESET],
    cwd=str(ROOT),
    check=True,
)

## 9 — Full pipeline (train-command)

Skips ingest / harmonize / preprocess / features; **loads baseline from disk** (`--skip-baseline`). Trains transformer with merged hparams.

Add **`--no-register`** if you do not want a second registry version after `transformer_tune` already registered the best run.

**MLflow persist:** remote URI was set in cell 2. For metrics-only full pipeline, add **`--mlflow-no-artifacts`** or set **`VINYL_GRADER_MLFLOW_METRICS_ONLY=1`** in cell 2 before `pipeline train`.

In [ ]:
%cd /content/vinyl_management_system
!python -m grader.src.pipeline train \
  --config /tmp/colab_grader_merged.yaml \
  --skip-ingest --skip-harmonize --skip-preprocess --skip-features \
  --skip-baseline

## 10 — Done

- **MLflow:** open your tracking server; experiment name from `grader.yaml` → `mlflow.experiment_name`.
- **Registry:** `mlflow.registry_model_name` in config (default often `VinylGrader`).
- **Sweep CSV:** `grader/reports/transformer_tune_results.csv`.
- **Artifact minimalism:** use `--mlflow-no-artifacts` or env **`VINYL_GRADER_MLFLOW_METRICS_ONLY=1`** on long exploratory `pipeline train` runs; rely on `transformer_tune --register` for the canonical model line.